In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path

import numpy as np
import pandas as pd

from relife.lifetime_model import SemiParametricAcceleratedFailureTime
from relife.lifetime_model._regression import LinearCovarEffect
from relife.lifetime_model._semi_parametric import SemiParamAFTData

In [ ]:
# Données chaines d'isolateur
relife_csv_datapath = Path(r"D:\Projets\RTE\ReLife\relife\relife\data\csv")
time, event, entry, *args = np.loadtxt(relife_csv_datapath / "insulator_string.csv", delimiter=",", skiprows=1,
                                       unpack=True)
covar = np.column_stack(args)

In [ ]:
# Données
channing_data = pd.read_csv(Path(r"D:\Projets\RTE\ReLife") / "channing.csv", sep=";", decimal=",")
channing_data = (
    channing_data
    .drop(columns="time")
    .rename(columns={"exit": "time", "cens": "event"})
)
time, event, entry = channing_data["time"].values, channing_data["event"].astype(float).values, channing_data["entry"].values
covar = (channing_data[["sex"]] == "Male").astype(float).values

In [ ]:
# Model
model = SemiParametricAcceleratedFailureTime()

In [ ]:
# Init covar_effect
N = len(covar)

model.covar_effect = LinearCovarEffect(
    (None,) * np.atleast_2d(np.asarray(covar, dtype=np.float64)).shape[-1]
)

# Build training_data
model._training_data = SemiParamAFTData(
    time=np.float64(time[:N]), covar=np.float64(covar[:N]), event=np.float64(event[:N]), entry=np.float64(entry[:N])
)

In [ ]:
# Log rank
model.log_rank_stat(np.zeros(3))

In [ ]:
# Test fit
model.fit(
     time=time[:N], covar=covar[:N], event=event[:N], entry=entry[:N]
)
print(model.params)